# EnzyMind Agent · 当前 Jupyter 接口验证

本 notebook 是 **2026-11-11 新建的独立验证记录**。原有 18 个 notebook 的代码、执行计数与保存输出均保留；不导入或执行任何旧案例。末尾会核验归档文件的 SHA-256。

验证范围：当前网站 Agent API、Jupyter 流式 Markdown、非流式调用、结果引用、持久会话恢复、FASTA 多记录和结果下载。科学查询仅使用酶注释及催化反应工具；不重新运行旧预测实验。

首次完整执行预计消耗 **3 次免费 Agent 机会**，与网站一样每天 10 次。固定 request ID 和本地持久会话别名使本 notebook 的相同请求可幂等重取。结果和对话在服务端保留 90 天，本 notebook 的输出与下载 JSON 可留档。

依赖安装到当前 notebook kernel：`python -m pip install -r agent_n8n/requirements-notebook.txt`。下方会自动定位项目目录。

In [1]:
from pathlib import Path
import hashlib
import json
import re
import sys
import uuid
from IPython.display import display, Markdown

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "agent_n8n/client.py").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from agent_n8n.client import EnzyMindN8NClient, EnzyMindAPIError

ARCHIVE = json.loads((ROOT / "tests/notebook_archive_manifest_20261111.json").read_text())
def verify_archive():
    changed = [item["path"] for item in ARCHIVE["notebooks"]
               if hashlib.sha256((ROOT / item["path"]).read_bytes()).hexdigest() != item["sha256"]]
    assert not changed, f"Archived notebooks changed: {changed}"
    return len(ARCHIVE["notebooks"])

print(f"项目：{ROOT.name}；原有 {verify_archive()} 个 notebook 校验通过。")


项目：Enzymind；原有 18 个 notebook 校验通过。


## 工作区身份与每日额度

私有身份文件保存在 Git 忽略的 `files/notebook-agent/workspace.json`，权限为 `0600`；请勿分享或提交该文件。新建客户端复用此身份，不会重置每日额度。

需要从 notebook 的结果链接进入网站时，在网站的 **Workspace access → Import** 导入这个私有文件即可看到同一工作区的结果。不要将访问 key 写入单元格或输出。

若当天免费额度已用完，先 `import getpass`，再对发起当前请求的客户端调用 `client.set_deepseek_key(getpass.getpass("DeepSeek API token: "))`。后续新建的 `restored`、`attachment_client` 也需通过 `deepseek_api_key` 参数或 `set_deepseek_key()` 设置；token 仅驻留内存，不会从身份文件恢复。`set_deepseek_key(None)` 可清除。

In [2]:
STATE = ROOT / "files/notebook-agent/workspace.json"
EXPORTS = ROOT / "files/notebook-agent/verification-20261111"
EXPORTS.mkdir(parents=True, exist_ok=True)
client = EnzyMindN8NClient(
    base_url=EnzyMindN8NClient.DEFAULT_BASE_URL,
    workspace_file=STATE,
    user_id="workspace-verification-20261111-annotations",
    timeout=180,
)
quota_before = client.usage()
print(f"API: {client.base_url}")
print(f"今日剩余免费机会：{quota_before['remaining']} / {quota_before['limit']}")
print(f"私有工作区文件（不展示 key）：{STATE.relative_to(ROOT)}")

# 相同会话、相同内容与 request ID 可重取结果，避免重跑本次验证计算。
def request_id(name):
    return str(uuid.uuid5(uuid.NAMESPACE_URL, "enzymind-notebook-workspace-verification-20261111/" + name))


API: https://cadd.zju.edu.cn/enzymind/api
今日剩余免费机会：8 / 10
私有工作区文件（不展示 key）：files/notebook-agent/workspace.json


## 1. 流式回答、工具调用与真实结果链接

查询 P00761 的酶注释和催化反应。保留原来的 `client.chat(...) -> str` 调用方式；`stream=True` 在 Jupyter 中更新同一块 Markdown。`on_tool_start/end` 的参数仍是工具序号。

In [3]:
tool_started, tool_finished, streamed_text = [], [], []
answer = client.chat(
    "请调用 Check Enzyme 和 Query Cat Activity 查询 UniProt P00761 的酶注释和催化反应。"
    "回答注明 EC 编号，并分别引用这两个工具返回的保存结果链接，不要调用预测模型。",
    stream=True,
    request_id=request_id("annotations"),
    on_tool_start=tool_started.append,
    on_tool_end=tool_finished.append,
    on_update=streamed_text.append,
)
assert client.last_turn["status"] == "completed"
assert "example.invalid" not in answer
assert "https://cadd." in answer and "/enzymind/results/" in answer
assert len(tool_started) == len(tool_finished) == 2
print(f"完成：已记录 {len(tool_finished)} 个工具调用；观测到 {client.stream_updates} 次流式文本更新。")
if client.stream_updates == 0:
    print("该请求已完成，通过固定 request ID 读取原答案，没有重新计算。")


**结论**：P00761 是酶，EC 编号为 **3.4.21.4**（胰蛋白酶类丝氨酸内肽酶），催化反应为对 Arg 和 Lys 羧基侧的优先切割：Arg-|-Xaa、Lys-|-Xaa。

**支持证据**

| 项目 | 结果 | 来源 |
|---|---|---|
| 是否为酶 | 是（数据库中检出催化活性注释） | [Check Enzyme 保存结果](https://cadd.zju.edu.cn/enzymind/results/9a43635d-c484-4ada-8b09-9145d5526260) |
| EC 编号 | 3.4.21.4 | [Check Enzyme 保存结果](https://cadd.zju.edu.cn/enzymind/results/9a43635d-c484-4ada-8b09-9145d5526260) |
| 催化反应 | Preferential cleavage: Arg-\|-Xaa, Lys-\|-Xaa（记录 1 条，Rhea ID 未提供） | [Query Cat Activity 保存结果](https://cadd.zju.edu.cn/enzymind/results/b1f6b0b9-51f8-427f-9537-7915e5817313) |

**说明**：两项结果均为数据库注释（非预测模型输出）。催化活动记录中 Rhea ID 为 N/A，即该条目未关联 Rhea 标识。

**建议下一步**：若需确认活性位点与机制细节，可查 P00761 的 UniProt 功能位点或 M-CSA 条目（如需要我可代为检索）。

完成：已记录 2 个工具调用；观测到 0 次流式文本更新。
该请求已完成，通过固定 request ID 读取原答案，没有重新计算。


In [4]:
session = client.get_session()
records = {r["tool"]: r for r in session["results"] if r["tool"] in ("check", "reaction")}
assert set(records) == {"check", "reaction"}
for record in records.values():
    assert record["status"] == "completed"
    detail = client.get_result(record["id"])
    assert detail["status"] == "completed"
    exported = client.export_result(record["id"], EXPORTS / f"{record['tool']}.json")
    print(f"{record['title']}: completed；已下载 {exported.relative_to(ROOT)}")

result_ids = {r["id"] for r in records.values()}
assert all(result_id in answer for result_id in result_ids)
saved_session_id = client.session_id
saved_turn_id = client.last_turn_id
print("结果链接已核验；完整结果已下载留档。")


Catalytic reactions: completed；已下载 files/notebook-agent/verification-20261111/reaction.json


Enzyme annotation: completed；已下载 files/notebook-agent/verification-20261111/check.json
结果链接已核验；完整结果已下载留档。


## 2. 恢复会话、只读恢复任务与非流式追问

新建客户端使用相同身份和 `session_id` 恢复持久记忆。`resume_turn()` 只读取原任务；随后用 `stream=False` 追问，复用已完成结果，不重复查询数据库。

In [5]:
restored = EnzyMindN8NClient(workspace_file=STATE, session_id=saved_session_id, timeout=180)
usage_before_resume = restored.usage()["used"]
replayed_answer = restored.resume_turn(saved_turn_id, display=False)
assert replayed_answer == answer
assert restored.usage()["used"] == usage_before_resume
assert restored.get_session()["id"] == saved_session_id

follow_up = restored.chat(
    "沿用刚才 P00761 的酶注释与催化反应结果，简洁复述 EC number，"
    "保留这两个已有结果的链接。不要调用工具或重复计算。",
    stream=False,
    request_id=request_id("follow-up"),
)
assert restored.last_turn["status"] == "completed"
assert not [e for e in restored.last_turn["events"] if e["kind"] == "tool_start"]
assert all(result_id in follow_up for result_id in result_ids)
assert "example.invalid" not in follow_up
print("新客户端恢复同一会话；任务重取未扣额度；追问未重复计算。")


**结论**：P00761 是酶，EC number 为 **3.4.21.4**（胰蛋白酶类丝氨酸内肽酶）；催化反应为优先切割 Arg 和 Lys 羧基侧：Arg-|-Xaa、Lys-|-Xaa。

| 项目 | 结果 | 来源 |
|---|---|---|
| 是否为酶 / EC 编号 | 是；3.4.21.4 | [Check Enzyme 保存结果](https://cadd.zju.edu.cn/enzymind/results/9a43635d-c484-4ada-8b09-9145d5526260) |
| 催化反应 | Preferential cleavage: Arg-\|-Xaa, Lys-\|-Xaa | [Query Cat Activity 保存结果](https://cadd.zju.edu.cn/enzymind/results/b1f6b0b9-51f8-427f-9537-7915e5817313) |

两项均为数据库注释结果（非预测模型输出）；催化活动记录未提供 Rhea ID。

新客户端恢复同一会话；任务重取未扣额度；追问未重复计算。


## 3. 兼容原来的 FASTA 文件参数

以下两条 **合成测试序列** 仅用于检查附件传输，不能用作科学结论。Agent 只检查条目名称与长度，不调用模型或数据库工具。每个 FASTA 条目保持独立。

In [6]:
fasta = EXPORTS / "synthetic_transport_fixture.fasta"
fasta.write_text('>transport_alpha\nACDEFGHIKLMNPQRSTVWY\n>transport_beta\nMKWVTFISLLFLFSSAYS\n', encoding="utf-8")
attachment_client = EnzyMindN8NClient(
    workspace_file=STATE, user_id="workspace-verification-20261111-fasta", timeout=180,
)
fasta_answer = attachment_client.chat(
    "这是纯接口传输测试，附件是合成序列。请只报告 FASTA 条目数量、"
    "各条目的原始名称和序列长度。不要把记录合并，不要调用任何工具或科学模型。",
    fasta_file=fasta,
    stream=True,
    request_id=request_id("fasta-transport"),
)
assert attachment_client.last_turn["status"] == "completed"
assert all(name in fasta_answer for name in ("transport_alpha", "transport_beta"))
assert not [e for e in attachment_client.last_turn["events"] if e["kind"] == "tool_start"]
assert "20" in fasta_answer and "18" in fasta_answer
print(f"FASTA 多条目传输完成；观测到 {attachment_client.stream_updates} 次流式文本更新；未调用科学计算工具。")


**FASTA 条目数量：2**

| 条目名称 | 序列长度 |
|---|---|
| transport_alpha | 20 |
| transport_beta | 18 |

各条目名称按原始写法保留，序列分别计数、未合并。按要求未调用任何工具或科学模型。

FASTA 多条目传输完成；观测到 3 次流式文本更新；未调用科学计算工具。


## 归档完整性与测试结论

本次不会修改或重执行旧案例。下方只校验原 notebook 文件，保存当前验证元数据；不输出工作区 key 或 DeepSeek token。

显式旧 Webhook 模式仍可使用：`EnzyMindN8NClient(webhook_url="http://localhost:5678/webhook/旧路径")`。该协议已通过独立 mock 回归测试；此 notebook 不触发旧工作流，也不迁移旧历史。

In [7]:
assert verify_archive() == 18
quota_after = client.usage()
summary = {
    "verification_date": "2026-11-11",
    "archived_notebooks_unchanged": verify_archive(),
    "annotations_completed": client.last_turn["status"] == "completed",
    "saved_results": len(records),
    "stream_updates_observed": {"annotations": client.stream_updates, "fasta": attachment_client.stream_updates},
    "restored_session": saved_session_id,
    "read_only_resume_preserved_quota": True,
    "follow_up_reused_results": True,
    "fasta_records_preserved": True,
    "free_turns_used_this_run": quota_after["used"] - quota_before["used"],
    "free_turns_remaining": quota_after["remaining"],
}
(EXPORTS / "verification.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2) + "\n")
print(json.dumps(summary, ensure_ascii=False, indent=2))
for connection in (client, restored, attachment_client):
    connection.close()


{
  "verification_date": "2026-11-11",
  "archived_notebooks_unchanged": 18,
  "annotations_completed": true,
  "saved_results": 2,
  "stream_updates_observed": {
    "annotations": 0,
    "fasta": 3
  },
  "restored_session": "29852687-cb36-4ab0-8ca2-e54333254f57",
  "read_only_resume_preserved_quota": true,
  "follow_up_reused_results": true,
  "fasta_records_preserved": true,
  "free_turns_used_this_run": 1,
  "free_turns_remaining": 7
}
